In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import numpy as np

## Реализация слоя прямого распространения

Выполняется аффинное преобразование: WX+b, задаётся количество нейронов на слое

In [ ]:
class Layer: # dense layer
    def __init__(self, input_size, output_size):
        self.weights = np.random.randn(input_size, output_size) * 0.01
        self.bias = np.zeros((1, output_size))
        self.grad_weights = None
        self.grad_bias = None
        self.last_input = None

    def forward(self, X):
        self.last_input = X
        return np.dot(X, self.weights) + self.bias

    def backward(self, grad_output):
        self.grad_weights = np.dot(self.last_input.T, grad_output)
        self.grad_bias = np.sum(grad_output, axis=0, keepdims=True)
        return np.dot(grad_output, self.weights.T)

## Реализация сети прямого распространения

Принимает список слоев

In [4]:
class NeuralNetwork:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, X):
        output = X
        for layer in self.layers:
            output = layer.forward(output)
        return output

    def backward(self, grad):
        current_grad = grad
        for layer in reversed(self.layers):
            current_grad = layer.backward(current_grad)

    def get_parameters(self):
        params = []
        for layer in self.layers:
            if isinstance(layer, Layer):
                params.append(layer.weights)
                params.append(layer.bias)
        return params

## Функции активации

In [5]:
class ReLU:
    def __init__(self):
        self.last_input = None

    def forward(self, X):
        self.last_input = X
        return np.maximum(0, X)

    def backward(self, grad_output):
        return grad_output * (self.last_input > 0)

In [6]:
class Sigmoid:
    def __init__(self):
        self.last_output = None

    def forward(self, X):
        self.last_output = 1 / (1 + np.exp(-X))
        return self.last_output

    def backward(self, grad_output):
        return grad_output * self.last_output * (1 - self.last_output)

## Оптимизатор (градиентный спуск)

In [7]:
class SGD:
    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate

    def update(self, params, grads):
        for param, grad in zip(params, grads):
            if grad is not None:
                param -= self.learning_rate * grad

## Функции потерь

### Кросс-энтропия

In [ ]:
class CrossEntropyLoss:
    def __init__(self):
        self.last_pred = None
        self.last_target = None
        self.last_target_indices = None

    def forward(self, pred, target):
        self.last_pred = pred
        self.last_target = target
    
        if target.ndim == 1:
            self.last_target_indices = target.astype(int)
        elif target.ndim == 2 and target.shape[1] == 1:
            self.last_target_indices = target.flatten().astype(int)
        else:
            self.last_target_indices = np.argmax(target, axis=1)
        
        log_probs = -np.log(pred[np.arange(len(pred)), self.last_target_indices])
        return np.mean(log_probs)

    def backward(self):
        if self.last_pred is None or self.last_target_indices is None:
            raise RuntimeError("Forward pass must be called before backward pass")
            
        grad = self.last_pred.copy()
        grad[np.arange(len(grad)), self.last_target_indices] -= 1
        return grad / len(grad)

## Загрузка данных

In [10]:
import pandas as pd

In [11]:
data = pd.read_csv("../data/processed_smoke_detector.csv")
X = data.drop(labels=["Fire Alarm"], axis=1)
y = data["Fire Alarm"]

In [12]:
from sklearn.feature_selection import SelectKBest

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
skb = SelectKBest(k=2)
X_skb = skb.fit_transform(X, y)
X_skb = pd.DataFrame(X_skb, columns=skb.get_feature_names_out())
X_skb.head()

,TVOC[ppb],Raw Ethanol
0,19.0,19951.0
1,1.0,19975.0
2,10.0,19955.0
3,10.0,19963.0
4,13.0,19958.0


In [15]:
X_skb.shape

(41247, 2)

In [16]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_skb, y, test_size=0.2, random_state=42)

In [17]:
num_features=2
num_classes=2

## Запуск сети

In [18]:
layers = [
    Layer(num_features, 64),
    ReLU(),
    Layer(64, 32),
    ReLU(),
    Layer(32, num_classes),
    Sigmoid()
]

In [19]:
model = NeuralNetwork(layers)
optimizer = SGD(learning_rate=0.1)
loss_fn = CrossEntropyLoss()

epochs = 1000
batch_size = 32
num_batches = int(np.ceil(X_train_clf.shape[0] / batch_size))

In [ ]:

for epoch in range(epochs):
    epoch_loss = 0
    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, X_train_clf.shape[0])
        X_batch = X_train_clf[start:end]
        y_batch = y_train_clf[start:end]
        
        output = model.forward(X_batch)
        loss = loss_fn.forward(output, y_batch)
        epoch_loss += loss

        grad = loss_fn.backward()
        model.backward(grad)
        
        linear_params = model.get_parameters()
        linear_grads = []
        for layer in model.layers:
            if isinstance(layer, Layer):
                linear_grads.extend([layer.grad_weights, layer.grad_bias])
        optimizer.update(linear_params, linear_grads)
    
    if (epoch + 1) % 100 == 0:
        avg_loss = epoch_loss / num_batches
        print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}')

Epoch 100/1000, Loss: 45.1549
Epoch 200/1000, Loss: 45.0897
Epoch 300/1000, Loss: 45.0192
Epoch 400/1000, Loss: 44.9425
Epoch 500/1000, Loss: 44.8584
Epoch 600/1000, Loss: 44.7653
Epoch 700/1000, Loss: 44.6610
Epoch 800/1000, Loss: 44.5425
Epoch 900/1000, Loss: 44.4053
Epoch 1000/1000, Loss: 44.2422


In [25]:
test_output      = model.forward(X_test_clf)
test_predictions = np.argmax(test_output, axis=1)
test_labels      = y_test_clf.to_numpy() 

In [24]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

accuracy = accuracy_score(test_labels, test_predictions)
precision = precision_score(test_labels, test_predictions)
recall = recall_score(test_labels, test_predictions)
f1 = f1_score(test_labels, test_predictions)

print(f"Accuracy: {accuracy:.2f}%")
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1-score:  {f1:.2f}")

Accuracy: 0.78%
Precision: 0.78
Recall:    1.00
F1-score:  0.87
